# Lab 01: Data Pipeline

## Business Context

The firm has S-1 filings from recent tech IPOs sitting in a Unity Catalog Volume (downloaded as HTML from SEC EDGAR), and stock return data in a Delta table. Before the analyzer can answer any questions, both data sources need to be parsed, chunked, indexed, and queryable.

**After this lab:** All filings are parsed into searchable chunks in a Delta table with a Vector Search index, and stock performance data is loaded into a structured Delta table. The foundation for everything that follows.

| Attribute | Detail |
|---|---|
| **Key Skills** | HTML text extraction, chunking, Delta Sync, managed embeddings, structured data ingestion |
| **Estimated Time** | 40 minutes |
| **Estimated Cost** | ~$3-4 |

In [ ]:
%pip install databricks-sdk databricks-vectorsearch mlflow langchain langchain-text-splitters beautifulsoup4 --quiet
dbutils.library.restartPython()

In [ ]:
CATALOG = "ipo_analyzer"
SCHEMA = "default"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/sec_filings"
LLM_ENDPOINT = "databricks-meta-llama-3.1-405b-instruct"
VS_ENDPOINT = "ipo_analyzer_vs_endpoint"

print(f"Catalog : {CATALOG}")
print(f"Volume  : {VOLUME_PATH}")

# List available filings
display(spark.sql(f"LIST '{VOLUME_PATH}'"))

## A. Parse S-1 Filings

Our S-1 filings are HTML files downloaded from SEC EDGAR. We extract text using BeautifulSoup, which strips HTML tags and returns clean text suitable for chunking and embedding.



In [ ]:
from bs4 import BeautifulSoup
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType
import re

def extract_text_from_html(html_bytes):
    """Extract clean text from SEC EDGAR HTML filing."""
    if html_bytes is None:
        return None
    try:
        html_str = html_bytes.decode("utf-8", errors="replace")
        soup = BeautifulSoup(html_str, "html.parser")
        # Remove script/style elements
        for tag in soup(["script", "style"]):
            tag.decompose()
        text = soup.get_text(separator="\n")
        # Collapse excessive whitespace while preserving paragraph breaks
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]+', ' ', text)
        lines = [line.strip() for line in text.split('\n')]
        text = '\n'.join(line for line in lines if line)
        return text
    except Exception as e:
        return f"ERROR: {e}"

extract_text_udf = udf(extract_text_from_html, StringType())

# Read all filings as binary files
docs_df = spark.read.format("binaryFile").load(VOLUME_PATH)
print(f"Found {docs_df.count()} filings")

# Parse HTML to text
parsed_df = docs_df.withColumn("filing_text", extract_text_udf(col("content")))

# Save to table (overwriteSchema needed since old table had VARIANT column)
PARSED_TABLE = f"{CATALOG}.{SCHEMA}.parsed_filings"
parsed_df.select("path", "filing_text").write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(PARSED_TABLE)
print(f"Parsed filings saved to {PARSED_TABLE}")

# Show results
display(spark.sql(f"""
SELECT
  SUBSTRING_INDEX(path, '/', -1) AS filing,
  LENGTH(filing_text) AS text_length,
  SUBSTRING(filing_text, 1, 200) AS preview
FROM {PARSED_TABLE}
ORDER BY text_length DESC
"""))

## B. Stock Performance Data

The stock performance data has been pre-loaded by the setup script (downloaded from Yahoo Finance locally, uploaded to Unity Catalog). This structured data lives alongside the unstructured S-1 text — the kind of cross-data join that no chat tool can do at scale.

The table contains IPO price, prices at 3/6/12 months post-IPO, and the 12-month return percentage for each company.

In [ ]:
# Stock performance data was pre-loaded by the setup script.
# Verify the table exists and inspect the data.
STOCK_TABLE = f"{CATALOG}.{SCHEMA}.stock_performance"

row_count = spark.table(STOCK_TABLE).count()
print(f"Stock performance table: {row_count} companies")
print(f"Table: {STOCK_TABLE}")

# Show the data sorted by 12-month return
display(spark.sql(f"""
SELECT company, ticker, sector, ipo_date, ipo_price, price_12m, twelve_month_return_pct
FROM {STOCK_TABLE}
ORDER BY twelve_month_return_pct DESC
"""))

# -- How the data was loaded (setup script): --
# The setup script downloads stock data locally via yfinance (Yahoo Finance
# Python library) and uploads it as a Delta table. yfinance doesn't work
# reliably on serverless compute (outbound HTTP restrictions), so this step
# runs locally before the labs.

## C. Chunk Filing Text

We split the extracted text into chunks suitable for embedding and vector search. Key parameters:
- **chunk_size = 3000**: Large chunks keep full sections together (e.g., an entire "Our Competition" section in one chunk)
- **chunk_overlap = 500**: Characters shared between adjacent chunks (prevents context loss at boundaries)

Larger chunks are critical for SEC filings where a single topic (competition, risk factors) spans 1000-3000 chars. Small chunks (1000) fragment these sections, making retrieval miss important context.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Read parsed filings
parsed = spark.table(f"{CATALOG}.{SCHEMA}.parsed_filings")

# Larger chunks keep full filing sections together (competition, risk factors, etc.)
CHUNK_SIZE = 3000
CHUNK_OVERLAP = 500

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Collect to driver for chunking (dataset is small enough — 19 filings)
raw_rows = parsed.select("path", "filing_text").collect()

chunks = []
for row in raw_rows:
    text = row.filing_text or ""
    if len(text) < 100:
        print(f"  WARN: {row.path} has very short text ({len(text)} chars), skipping")
        continue
    text_chunks = text_splitter.split_text(text)
    for i, chunk in enumerate(text_chunks):
        chunks.append((row.path, i, chunk))

print(f"Created {len(chunks)} chunks from {len(raw_rows)} filings")

schema = StructType([
    StructField("path", StringType()),
    StructField("chunk_index", IntegerType()),
    StructField("chunk_text", StringType()),
])
chunks_df = spark.createDataFrame(chunks, schema)

# Add unique chunk ID (required for Vector Search)
chunks_flat_df = chunks_df.withColumn(
    "chunk_id",
    F.sha2(F.concat(F.col("path"), F.lit("_"), F.col("chunk_index").cast("string")), 256)
).select("chunk_id", "path", "chunk_index", "chunk_text")

print(f"Total chunks: {chunks_flat_df.count()}")

In [ ]:
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.filing_chunks"

(
    chunks_flat_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(CHUNKS_TABLE)
)

print(f"Saved to {CHUNKS_TABLE}")

# Show chunks per filing
display(spark.sql(f"""
SELECT
  SUBSTRING_INDEX(path, '/', -1) AS filing,
  COUNT(*) AS chunks,
  ROUND(AVG(LENGTH(chunk_text))) AS avg_chunk_len
FROM {CHUNKS_TABLE}
GROUP BY path
ORDER BY chunks DESC
"""))

## D. Create Vector Search Index

A **Vector Search Endpoint** hosts the compute. A **Delta Sync Index** with **Managed Embeddings** auto-generates embeddings using `databricks-bge-large-en` and stays in sync with the source Delta table via Change Data Feed.

Prerequisites (already done):
1. Source table has a unique primary key (`chunk_id`)
2. Source table has Change Data Feed enabled

In [ ]:
from databricks.vector_search.client import VectorSearchClient
import time

vsc = VectorSearchClient()

VS_ENDPOINT = "ipo_analyzer_vs_endpoint"
VS_INDEX = f"{CATALOG}.{SCHEMA}.filing_chunks_index"

# Create or reuse existing endpoint
try:
    vsc.create_endpoint(name=VS_ENDPOINT, endpoint_type="STANDARD")
    print(f"Creating endpoint '{VS_ENDPOINT}'...")
except Exception as e:
    if "already exists" in str(e).lower() or "quota" in str(e).lower():
        print(f"Endpoint '{VS_ENDPOINT}' already exists, reusing.")
    else:
        raise

# Wait for endpoint to be online
for i in range(30):
    try:
        ep = vsc.get_endpoint(VS_ENDPOINT)
        status = ep.get("endpoint_status", {}).get("state", "UNKNOWN")
        print(f"  Endpoint status: {status}")
        if status == "ONLINE":
            break
    except Exception:
        pass
    time.sleep(10)

print(f"Endpoint '{VS_ENDPOINT}' ready.")

In [ ]:
# Delete existing index if it has stale data (0 chunks from previous run)
try:
    idx = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)
    print(f"Index '{VS_INDEX}' exists. Deleting to rebuild with new chunks...")
    vsc.delete_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)
    print("Deleted. Waiting 10s before recreating...")
    time.sleep(10)
except Exception as e:
    if "not found" in str(e).lower() or "does not exist" in str(e).lower():
        print(f"No existing index. Creating fresh.")
    else:
        print(f"Note: {e}")

# Create Delta Sync index with managed embeddings
index = vsc.create_delta_sync_index(
    endpoint_name=VS_ENDPOINT,
    index_name=VS_INDEX,
    source_table_name=f"{CATALOG}.{SCHEMA}.filing_chunks",
    pipeline_type="TRIGGERED",
    primary_key="chunk_id",
    embedding_source_column="chunk_text",
    embedding_model_endpoint_name="databricks-bge-large-en"
)
print(f"Creating index '{VS_INDEX}'...")

# Wait for index to sync
print("Waiting for index to sync (may take several minutes)...")
for i in range(60):
    try:
        idx = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)
        status = idx.describe().get("status", {})
        ready = status.get("ready", False)
        msg = status.get("message", "")
        print(f"  Index: ready={ready} | {msg}")
        if ready:
            break
    except Exception as e:
        print(f"  Checking: {e}")
    time.sleep(15)

print(f"Index '{VS_INDEX}' ready.")

In [ ]:
# Test: semantic search for "competition"
idx = vsc.get_index(VS_ENDPOINT, VS_INDEX)
results = idx.similarity_search(
    query_text="competitive landscape and market position",
    columns=["chunk_id", "path", "chunk_text"],
    num_results=3,
    query_type="HYBRID",
)

print("=== Hybrid Search Results ===")
for doc in results.get("result", {}).get("data_array", []):
    source = doc[1].split("/")[-1].replace("-S1.html", "")
    print(f"\nSource: {source}")
    print(f"  {doc[2][:200]}...")

## Before / After

**Before this lab:** S-1 filings are raw HTML files in a volume. Stock data is on Yahoo Finance. To find "what did Snowflake say about competition?", you'd open each filing manually and Ctrl+F.

**After this lab:** Both data sources are in Delta tables. You can find relevant passages in milliseconds via Vector Search, and cross-reference with stock performance data.

In [ ]:
import pyspark.sql.functions as F

# BEFORE: SQL LIKE scan -- slow, misses synonyms
print("=== BEFORE: SQL LIKE scan ===")
like_results = spark.sql(f"""
SELECT path, chunk_text
FROM {CATALOG}.{SCHEMA}.filing_chunks
WHERE LOWER(chunk_text) LIKE '%competitive%'
LIMIT 3
""")
print(f"SQL LIKE 'competitive': {like_results.count()} results (exact match only)")
display(like_results.select("path", F.substring("chunk_text", 1, 150).alias("preview")))

print()

# AFTER: Vector Search -- fast, semantic
print("=== AFTER: Vector Search (semantic) ===")
vs_results = idx.similarity_search(
    query_text="competitive landscape and market position",
    columns=["path", "chunk_text"],
    num_results=3,
    query_type="HYBRID",
)
vs_docs = vs_results.get("result", {}).get("data_array", [])
print(f"Vector Search 'competitive landscape': {len(vs_docs)} results (semantic match)")
for doc in vs_docs:
    source = doc[0].split("/")[-1].replace("-S1.html", "")
    print(f"  [{source}] {doc[1][:150]}...")

print()
print("Stock data preview:")
display(spark.sql(f"""
SELECT company, ticker, twelve_month_return_pct
FROM {CATALOG}.{SCHEMA}.stock_performance
ORDER BY twelve_month_return_pct DESC
LIMIT 5
"""))